In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# EDIT THESE PATHS
ES_PATH = "outputs/vicuna/<es_run>/final_results/results.csv"
GA_PATH = "outputs/vicuna/<ga_run>/final_results/results.csv"
PSO_H_PATH = "outputs/vicuna/<pso_hybrid_run>/final_results/results.csv"
PSO_LLM_PATH = "outputs/vicuna/<pso_llm_run>/final_results/results.csv"

In [ ]:
from pathlib import Path

ROOT = Path('..').resolve()
REQUIRED_COLS = [
    'query_id', 'attack_success', 'fitness', 'alignment_score',
    'generation', 'generation_reached'
]

def _safe_path(path: str) -> Path:
    p = Path(path)
    if p.is_absolute():
        return p
    return ROOT / p

def load_df(path, name):
    full_path = _safe_path(path)
    if not full_path.exists():
        print(f'[WARN] Missing file for {name}: {full_path}')
        df = pd.DataFrame(columns=REQUIRED_COLS)
    else:
        df = pd.read_csv(full_path)

    if 'query_id' not in df.columns:
        df['query_id'] = pd.Series(dtype='object')

    for col in ['attack_success', 'fitness', 'alignment_score', 'generation', 'generation_reached']:
        if col not in df.columns:
            df[col] = np.nan

    df['query_id'] = (
        df['query_id']
        .astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )
    df['algo'] = name
    return df

es = load_df(ES_PATH, 'ES')
ga = load_df(GA_PATH, 'GA')
pso_h = load_df(PSO_H_PATH, 'PSO-Hybrid')
pso_l = load_df(PSO_LLM_PATH, 'PSO-LLM')

print('Rows loaded:', {'ES': len(es), 'GA': len(ga), 'PSO-Hybrid': len(pso_h), 'PSO-LLM': len(pso_l)})

In [ ]:
common = set(es['query_id']) & set(ga['query_id']) & set(pso_h['query_id']) & set(pso_l['query_id'])
common = {q for q in common if q not in {'', 'nan', 'None'}}

print('Common queries:', len(common))

def align(df):
    out = df[df['query_id'].isin(common)].copy()
    out['_qid_num'] = pd.to_numeric(out['query_id'], errors='coerce')
    out = out.sort_values(['_qid_num', 'query_id']).drop(columns=['_qid_num']).reset_index(drop=True)
    return out

es = align(es)
ga = align(ga)
pso_h = align(pso_h)
pso_l = align(pso_l)

assert len(es) == len(ga) == len(pso_h) == len(pso_l), 'Alignment mismatch after query_id intersection'
if len(es) > 0:
    assert all(es['query_id'] == ga['query_id'])
    assert all(es['query_id'] == pso_h['query_id'])
    assert all(es['query_id'] == pso_l['query_id'])

print('Aligned rows per algorithm:', len(es))

In [ ]:
def _parse_success(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])

def summary(df):
    if df.empty:
        return {'success': 0.0, 'fitness': 0.0, 'alignment': 0.0, 'rows': 0}

    fitness = pd.to_numeric(df['fitness'], errors='coerce')
    alignment = pd.to_numeric(df['alignment_score'], errors='coerce')
    success = _parse_success(df['attack_success'])

    return {
        'success': float(success.mean()) if len(success) else 0.0,
        'fitness': float(fitness.mean(skipna=True)) if fitness.notna().any() else 0.0,
        'alignment': float(alignment.mean(skipna=True)) if alignment.notna().any() else 0.0,
        'rows': int(len(df)),
    }

summary_df = pd.DataFrame([
    ('ES', summary(es)),
    ('GA', summary(ga)),
    ('PSO-Hybrid', summary(pso_h)),
    ('PSO-LLM', summary(pso_l)),
], columns=['algo', 'metrics'])

summary_df = pd.concat([summary_df['algo'], summary_df['metrics'].apply(pd.Series)], axis=1)

print(summary_df)

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x='algo', y='success', data=summary_df)
plt.title('Attack Success Rate')
plt.xlabel('Algorithm')
plt.ylabel('Success Rate')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
if len(es):
    sns.histplot(pd.to_numeric(es['fitness'], errors='coerce').dropna(), label='ES', kde=True, stat='density', alpha=0.4)
if len(ga):
    sns.histplot(pd.to_numeric(ga['fitness'], errors='coerce').dropna(), label='GA', kde=True, stat='density', alpha=0.4)
if len(pso_h):
    sns.histplot(pd.to_numeric(pso_h['fitness'], errors='coerce').dropna(), label='PSO-H', kde=True, stat='density', alpha=0.4)
if len(pso_l):
    sns.histplot(pd.to_numeric(pso_l['fitness'], errors='coerce').dropna(), label='PSO-LLM', kde=True, stat='density', alpha=0.4)
plt.legend()
plt.title('Fitness Distribution')
plt.xlabel('Fitness')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
es_fit = pd.to_numeric(es['fitness'], errors='coerce')
ga_fit = pd.to_numeric(ga['fitness'], errors='coerce')
valid = es_fit.notna() & ga_fit.notna()
plt.scatter(es_fit[valid], ga_fit[valid], alpha=0.5)
plt.xlabel('ES')
plt.ylabel('GA')
plt.title('ES vs GA Fitness')
plt.tight_layout()
plt.show()

In [ ]:
def _best_by_generation(df: pd.DataFrame) -> pd.Series:
    gen_col = None
    for col in ['generation', 'generation_reached']:
        if col in df.columns and pd.to_numeric(df[col], errors='coerce').notna().any():
            gen_col = col
            break

    if gen_col is None or df.empty:
        return pd.Series(dtype=float)

    tmp = df.copy()
    tmp[gen_col] = pd.to_numeric(tmp[gen_col], errors='coerce')
    tmp['fitness_num'] = pd.to_numeric(tmp['fitness'], errors='coerce')
    tmp = tmp.dropna(subset=[gen_col, 'fitness_num'])
    if tmp.empty:
        return pd.Series(dtype=float)

    tmp[gen_col] = tmp[gen_col].astype(int)
    return tmp.groupby(gen_col)['fitness_num'].max().sort_index()

es_curve = _best_by_generation(es)
ga_curve = _best_by_generation(ga)
pso_h_curve = _best_by_generation(pso_h)
pso_l_curve = _best_by_generation(pso_l)

if es_curve.empty and ga_curve.empty and pso_h_curve.empty and pso_l_curve.empty:
    print('No generation data available in final results files.')
else:
    plt.figure(figsize=(8, 4))
    if not es_curve.empty:
        plt.plot(es_curve.index, es_curve.values, label='ES')
    if not ga_curve.empty:
        plt.plot(ga_curve.index, ga_curve.values, label='GA')
    if not pso_h_curve.empty:
        plt.plot(pso_h_curve.index, pso_h_curve.values, label='PSO-H')
    if not pso_l_curve.empty:
        plt.plot(pso_l_curve.index, pso_l_curve.values, label='PSO-LLM')
    plt.legend()
    plt.title('Best Fitness per Generation')
    plt.xlabel('Generation')
    plt.ylabel('Best Fitness')
    plt.tight_layout()
    plt.show()

## Interpretation

- Which algorithm has highest success?
- Which converges fastest?
- Does PSO-LLM outperform hybrid?
- Is GA more unstable than ES?